# 04 数据库速查
直接用 SQL 看 `bars` 表。

In [1]:
import pandas as pd, sqlalchemy as sa
from ibkr_quant.db import engine
def q(sql, **params):
    return pd.read_sql(sa.text(sql), engine, params=params)

In [2]:
q('select symbol, bar_size, count(*) as rows, min(ts)::date as first, max(ts)::date as last from bars group by 1,2 order by 1')

,symbol,bar_size,rows,first,last
0,GLD,1 day,1969,2019-01-02,2026-10-30
1,IWM,1 day,1969,2019-01-02,2026-10-30
2,QQQ,1 day,1969,2019-01-02,2026-10-30
3,SPY,1 day,1969,2019-01-02,2026-10-30
4,TLT,1 day,1969,2019-01-02,2026-10-30


In [3]:
q("select ts::date as date, open, high, low, close, volume from bars where symbol = :sym order by ts desc limit 10", sym='SPY')

,date,open,high,low,close,volume
0,2026-10-30,761.38,761.46,750.36,755.72,49558926.0
1,2026-10-29,768.80,770.76,766.62,769.36,24186119.0
2,2026-10-28,771.43,773.36,769.24,772.31,19642942.0
3,2026-10-27,767.34,771.14,765.21,769.66,19753852.0
4,2026-10-26,762.25,767.73,761.63,765.37,20719600.0
5,2026-10-23,759.90,763.47,758.68,759.97,21143828.0
6,2026-10-22,766.37,767.71,761.16,763.96,34401213.0
7,2026-10-21,771.18,774.44,768.40,773.09,22682186.0
8,2026-10-20,765.87,772.29,764.25,769.05,25315354.0
9,2026-10-19,767.23,770.46,765.82,766.48,25219095.0


In [4]:
# 单日最大涨跌
q("""select * from (
         select symbol, ts::date as date,
                round((close / lag(close) over (partition by symbol order by ts) - 1)::numeric * 100, 2) as ret_pct
         from bars where bar_size = '1 day') t
       where ret_pct is not null order by abs(ret_pct) desc limit 10""")

,symbol,date,ret_pct
0,IWM,2020-03-16,-13.27
1,QQQ,2025-04-09,12.00
2,QQQ,2020-03-16,-11.98
3,IWM,2020-03-12,-11.05
4,SPY,2020-03-16,-10.94
5,SPY,2025-04-09,10.50
6,GLD,2026-01-30,-10.27
7,IWM,2020-03-09,-9.72
8,SPY,2020-03-12,-9.57
9,QQQ,2020-03-12,-9.17
